# 問題ロールアウトの動画作成

`collect_and_analyze.ipynb` で抽出したロールアウトを、次のパネルを並べた動画にする。

- 真横視点のレンダリング
- リストカメラの映像（方策が見ている向き）
- 手先の高さの実績と、その時点のチャンクから計算した想定軌道
- その時点のチャンクの z 指令とグリッパー指令

記録済みのアクションをシミュレータで再生するので、モデルは使わない。

VM の起動と実行の手順は `doc/colab_vm.md` を参照する。実行前に `/content/rclone.conf` をアップロードしておく。

## 1. セットアップ

In [ ]:
# VM の cmake は pip の隔離ビルドで動かないため、システムの cmake を先に使う
!apt-get install -y -q cmake > /dev/null
!PATH=/usr/bin:$PATH pip install -q "lerobot[pi,libero]==0.6.1" imageio-ffmpeg
!curl -fsSL https://rclone.org/install.sh | bash > /dev/null

In [ ]:
import importlib.util
import os
import pathlib

import yaml

# ヘッドレス環境で MuJoCo を GPU 描画する
os.environ["MUJOCO_GL"] = "egl"

# Colab の VM には NVIDIA の EGL 定義がない場合があるため補う
icd = pathlib.Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

# libero は設定ファイルがないと import 時に input() で止まるため、先に作っておく
libero_root = pathlib.Path(importlib.util.find_spec("libero.libero").origin).parent
config_file = pathlib.Path.home() / ".libero/config.yaml"
config_file.parent.mkdir(exist_ok=True)
config_file.write_text(yaml.dump({
    "benchmark_root": str(libero_root),
    "bddl_files": str(libero_root / "bddl_files"),
    "init_states": str(libero_root / "init_files"),
    "datasets": str(libero_root.parent / "datasets"),
    "assets": str(libero_root / "assets"),
}))

## 2. 設定とデータ取得

`NUM_VIDEOS` 件を、誤差の大きい順に動画にする。`ARM` を指定すると、`atomic_compare.ipynb` の結果を対象にする。

In [ ]:
import subprocess

ARM = "d16"  # 対策ありの条件。None なら対策なしの動画を作る
NUM_VIDEOS = 5
GAIN = 0.0114  # collect_and_analyze.ipynb で実測した換算係数
SEED = 7  # 収集時と同じ値にする
IMAGE_SIZE = 256  # 収集時と同じ値にする
SETTLE_STEPS = 10  # 収集時と同じ値にする
DUMMY_ACTION = [0, 0, 0, 0, 0, 0, -1]
FPS = 10  # 実時間の半分の速度
SIDE_CAMERA_DISTANCE_M = 0.8  # 対象物から真横のカメラまでの距離
SIDE_CAMERA_CLIP_M = 0.55  # カメラからこの距離より手前は描画しない
LIFT_CUT_M = 0.01  # 把持後に手先がこれだけ上がったら動画を終える

RCLONE_CONF = "/content/rclone.conf"
REMOTE_DIR = "remote:pai_homework2/approach_analysis"
OUT_DIR = pathlib.Path("/content/videos")
OUT_DIR.mkdir(exist_ok=True)
EPISODE_DIR = pathlib.Path("/content/results/episodes" if ARM is None else f"/content/atomic/{ARM}")
ROLLOUTS_CSV = "/content/results/problem_rollouts.csv" if ARM is None else f"/content/atomic/{ARM}/video_rollouts.csv"
KEY_LABEL = "worst" if ARM is None else "frozen"  # グラフで固定表示するチャンクの呼び方

# 収集結果を取得する
subprocess.run(["rclone", "copyto", f"{REMOTE_DIR}/results.tar.gz", "/content/results.tar.gz",
                "--config", RCLONE_CONF, "--retries", "10"], check=True)
subprocess.run(["tar", "xzf", "/content/results.tar.gz", "-C", "/content"], check=True)
if ARM is not None:
    subprocess.run(["rclone", "copyto", f"{REMOTE_DIR}/atomic.tar.gz", "/content/atomic.tar.gz",
                    "--config", RCLONE_CONF, "--retries", "10"], check=True)
    subprocess.run(["tar", "xzf", "/content/atomic.tar.gz", "-C", "/content"], check=True)

## 3. 再生

収集時と同じ種と初期状態から、記録したアクションを順に実行する。

真横のカメラは、`frontview` を対象物の -y 側へ動かして +y 方向へ向ける。x 軸方向の位置と高さが見えるので、手先と対象物の高さ関係を確認できる。

In [ ]:
import random
import zlib

import numpy as np
import torch
from lerobot.envs.libero import get_task_init_states
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv


def rollout_seed(*key):
    """rollout.py と同じ方法で種を作る。"""
    return zlib.crc32("/".join(map(str, (SEED, *key))).encode())


def replay(suite_name, task_id, episode, data):
    """記録したアクションを再生し、真横視点とリストカメラの画像、手先の高さのずれを返す。"""
    suite = benchmark.get_benchmark_dict()[suite_name]()
    task = suite.get_task(task_id)
    bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env = OffScreenRenderEnv(bddl_file_name=str(bddl_file), camera_heights=IMAGE_SIZE, camera_widths=IMAGE_SIZE)

    seed = rollout_seed(suite_name, task_id, episode)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    env.seed(seed)
    env.reset()
    init_states = get_task_init_states(suite, task_id)
    obs = env.set_init_state(init_states[episode % len(init_states)])
    for _ in range(SETTLE_STEPS):
        obs, _, _, _ = env.step(DUMMY_ACTION)

    # 真横のカメラを、対象物の -y 側に水平に置く
    sim = env.env.sim
    camera = sim.model.camera_name2id("frontview")
    target = data["step_obj"][0]
    sim.model.cam_pos[camera] = target + np.array([0.0, -SIDE_CAMERA_DISTANCE_M, 0.1])
    sim.model.cam_quat[camera] = [np.sqrt(0.5), np.sqrt(0.5), 0.0, 0.0]  # (w, x, y, z)。x 軸まわりに 90 度
    # キャビネットなどで遮られないよう、真横の描画のときだけカメラの手前の物体を外す
    default_znear = sim.model.vis.map.znear

    side_images, wrist_images, drifts = [], [], []
    for t, action in enumerate(data["step_action"]):
        sim.model.vis.map.znear = SIDE_CAMERA_CLIP_M / sim.model.stat.extent
        side_images.append(sim.render(camera_name="frontview", width=480, height=480)[::-1])
        sim.model.vis.map.znear = default_znear
        wrist_images.append(obs["robot0_eye_in_hand_image"][::-1, ::-1].copy())
        drifts.append(abs(obs["robot0_eef_pos"][2] - data["step_eef"][t, 2]))
        obs, _, _, _ = env.step(action)
    env.close()
    return side_images, wrist_images, np.array(drifts)

## 4. 動画の描画

把持後は対象物も一緒に持ち上がり、高さの差が意味を持たなくなるので、手先が上がり始めたところで動画を終える。

右上のグラフは、手先と対象物の高さの差である。黒線が実績、青の破線がその時点のチャンクから計算した想定軌道、灰色の線がタスクごとの基準の高さを表す。

想定軌道は `推論時の高さ + 換算係数 × z アクションの累積` で計算する。z アクションは下降が負である。実績が止まっているのに想定軌道が下がり続ければ、指令が実際の動きになっていないことを示す。

誤差が最大だった推論、または対策が発動した推論のチャンクは赤で固定表示する。

In [ ]:
import imageio.v2 as imageio
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


def lift_start(data):
    """把持後に手先が上がり始めたステップを返す。"""
    heights = data["step_eef"][:, 2]
    close = int(np.flatnonzero(data["step_action"][:, 6] > 0)[0])
    # 配置時の下降を拾わないよう、閉指令以降の最低点から最初に上がった点を探す
    lowest_so_far = np.minimum.accumulate(heights[close:])
    lifted = np.flatnonzero(heights[close:] > lowest_so_far + LIFT_CUT_M)
    return close + int(lifted[0]) if len(lifted) else len(heights) - 1


def render_video(row, path):
    """1 ロールアウトの動画を保存し、再生のずれの最大値を返す。"""
    file = f"{row.suite}_task{row.task_id:02d}_ep{row.episode:03d}.npz"
    with np.load(EPISODE_DIR / file) as npz:
        data = {key: npz[key] for key in npz.files}
    side_images, wrist_images, drifts = replay(row.suite, row.task_id, row.episode, data)

    heights = data["step_eef"][:, 2] - data["step_obj"][:, 2]
    infer_steps = data["infer_t"]
    # 基準の高さは、誤差が最大だった推論の必要下降距離から逆算する
    level = heights[row.t] - row.required
    worst_index = list(infer_steps).index(row.t)
    worst_plan = heights[row.t] + GAIN * np.cumsum(data["chunks"][worst_index, :, 2])
    steps = np.arange(len(heights))
    plan_steps = np.arange(data["chunks"].shape[1])

    fig = plt.figure(figsize=(12.8, 7.2), dpi=100)
    grid = fig.add_gridspec(2, 3)
    side_ax = fig.add_subplot(grid[0, 0])
    wrist_ax = fig.add_subplot(grid[1, 0])
    height_ax = fig.add_subplot(grid[0, 1:])
    chunk_ax = fig.add_subplot(grid[1, 1:])

    last = lift_start(data)
    with imageio.get_writer(path, fps=FPS, codec="libx264", macro_block_size=1) as writer:
        for t in range(last + 1):
            index = np.searchsorted(infer_steps, t, side="right") - 1
            chunk = data["chunks"][index]
            start = infer_steps[index]
            plan = heights[start] + GAIN * np.cumsum(chunk[:, 2])

            for ax in [side_ax, wrist_ax, height_ax, chunk_ax]:
                ax.clear()
            side_ax.imshow(side_images[t])
            side_ax.set_title("Side view")
            wrist_ax.imshow(wrist_images[t])
            wrist_ax.set_title("Wrist camera (policy input)")
            side_ax.axis("off")
            wrist_ax.axis("off")

            height_ax.plot(steps[:t + 1], heights[:t + 1], color="black", label="Actual")
            height_ax.plot(start + plan_steps, plan, "--", color="tab:blue", label="Current chunk plan")
            height_ax.plot(row.t + plan_steps, worst_plan, ":", color="tab:red", label=f"{KEY_LABEL.capitalize()} chunk (t={row.t})")
            height_ax.axhline(level, color="gray", linewidth=1, label="Reference grasp height")
            height_ax.set_xlim(0, last)
            height_ax.set_ylim(min(worst_plan.min(), level) - 0.03, heights.max() + 0.03)
            height_ax.set_xlabel("Step")
            height_ax.set_ylabel("Hand height above object [m]")
            height_ax.set_title(f"{row.suite} task {row.task_id} ep {row.episode}  (0.5x speed)  "
                                f"{KEY_LABEL} t={row.t}: required {row.required * 100:.1f} cm, "
                                f"planned {row.planned * 100:.1f} cm")
            height_ax.legend(loc="upper right", fontsize=8)

            chunk_ax.plot(start + plan_steps, chunk[:, 2], color="tab:blue", label="Chunk z command")
            chunk_ax.plot(start + plan_steps, chunk[:, 6], color="tab:orange", label="Chunk gripper command")
            chunk_ax.axvline(t, color="black", linewidth=1)
            chunk_ax.set_xlim(0, last)
            chunk_ax.set_ylim(-1.2, 1.2)
            chunk_ax.set_xlabel("Step")
            chunk_ax.set_ylabel("Command")
            chunk_ax.legend(loc="upper right", fontsize=8)

            fig.tight_layout()
            fig.canvas.draw()
            writer.append_data(np.asarray(fig.canvas.buffer_rgba())[:, :, :3])
    plt.close(fig)
    return drifts.max()

## 5. 動画の作成

再生の再現性を確認するため、記録と再生の手先の高さのずれも表示する。

In [ ]:
import pandas as pd

problems = pd.read_csv(ROLLOUTS_CSV)

for rank, row in enumerate(problems.head(NUM_VIDEOS).itertuples()):
    path = OUT_DIR / f"{rank:02d}_{row.suite}_task{row.task_id:02d}_ep{row.episode:03d}.mp4"
    drift = render_video(row, path)
    print(path.name, f"再生のずれ: {drift * 1000:.3f} mm", flush=True)

subprocess.run(["rclone", "copy", str(OUT_DIR), f"{REMOTE_DIR}/videos_{ARM or 'none'}", "--config", RCLONE_CONF, "--retries", "10"], check=True)

## 6. 後始末

Drive の認証情報を VM から削除する。VM の停止は手元から `colab stop` で行う。

In [ ]:
!shred -u /content/rclone.conf